# 第5日 ノートブック02：PatchCore を学習させる

# Day 5 Notebook 02: Training PatchCore

---

## 学習目標 / Learning objectives

| 日本語 | English |
| --- | --- |
| 1. Anomalib の API を使って PatchCore を初期化できる | 1. Initialise PatchCore through the Anomalib API |
| 2. 異常検知用のデータ構成を説明できる | 2. Explain the dataset layout used for anomaly detection |
| 3. 正常画像だけで学習（fit）できる | 3. Fit the model on normal images only |
| 4. 異常スコアとヒートマップを取得できる | 4. Obtain anomaly scores and heatmaps |
| 5. 結果を保存し、しきい値を検討できる | 5. Save results and reason about the threshold |

## このノートブックの基本情報 / Notebook facts

| 項目 / Item | 内容 / Details |
| --- | --- |
| 学習時間（目安） / Estimated duration | 130 分 / 130 minutes |
| 難易度 / Difficulty | 初級 / Beginner |
| 前提条件 / Prerequisites | 第5日ノートブック01 を終え、`aivision-anomalib` カーネルを選択していること。 / You finished Day 5 Notebook 01 and selected the `aivision-anomalib` kernel. |
| 必要なライブラリ / Required libraries | anomalib 2.7.0, torch, torchvision, opencv-python-headless, numpy, matplotlib, pandas |
| 使用カーネル / Jupyter kernel | `aivision-anomalib`（**必ずこのカーネルを使ってください**） |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第5日 / Day 5](../../COURSE_SYLLABUS.md#day5) |

### 提出物 / Expected deliverables

| 日本語 | English |
| --- | --- |
| 学習結果（指標） | Training results (metrics) |
| ヒートマップ画像 | Heatmap images |
| 画像ごとの異常スコアの表 | A table of per-image anomaly scores |
| 画像ごとの数値マップ（maps/*.npz） | A numeric anomaly map per image (maps/*.npz) |

> This notebook **requires the `aivision-anomalib` kernel**. Week 1 uses two separate environments because Anomalib requires `opencv-python-headless` while Ultralytics requires `opencv-python`; installing both into one environment breaks OpenCV.

## このノートブックの位置 / Where this notebook sits

| 項目 / Item | 内容 / Details |
| --- | --- |
| 日 / Day | 第5日 / Day 5 |
| この日の順番 / Position today | 2 / 3（この日のノートブックのうち）/ of today's notebooks |
| 所要時間 / Duration | 130 分 / 130 minutes |
| 前提 / Prerequisites | `Day05_01_Anomaly_Detection_Concepts.ipynb`、`aivision-anomalib` カーネル |
| この日の時間割 / Today's timetable | [`COURSE_SYLLABUS.md` 第5日 / Day 5](../../COURSE_SYLLABUS.md#day5) |

1 日の時間割は `COURSE_SYLLABUS.md` に 1 つだけあります。このノートブックはその一部で、
自分の時間割を持ちません。時刻が分からなくなったら、まずそこを見てください。

The day's timetable exists once, in `COURSE_SYLLABUS.md`. This notebook is one part of it and
does not keep its own schedule; look there first when you are unsure of the time.


### 予想して、実行して、説明する / Predict, run, inspect, explain

バックボーンは固定した特徴抽出器です。正常メモリと較正値はデータから作ります。 重要なセルの前に、入力・操作・出力を一文ずつ説明してください。
予想を書いて実行し、値または図を観察します。較正用の条件だけ（試験は調整に使わない）を変え、もう一度予想して実行します。
意図的なエラーと未完成の演習は、その指示に従って取り組みます。

The backbone is a frozen feature extractor; data determine normal memory and calibration. Before an important cell, describe its input, operation and output in one sentence each.
Predict, run, inspect the values/image and explain why. Change only a calibration setting (never tune on test), predict and rerun.
Follow the instructions for deliberately erroneous cells and unsolved exercises.


---

## Part 0　準備 / Preparation

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from pathlib import Path
import sys
import os
os.environ["TIMM_USE_OLD_CACHE"] = "1"

COURSE_ROOT = Path.cwd().resolve()
while not ((COURSE_ROOT / "assets").is_dir() and (COURSE_ROOT / "notebooks").is_dir()):
    if COURSE_ROOT.parent == COURSE_ROOT:
        raise FileNotFoundError(
            "研修フォルダが見つかりません。JupyterLab を AI_Vision_Training で起動してください。\n"
            "Course folder not found. Start JupyterLab inside the AI_Vision_Training folder."
        )
    COURSE_ROOT = COURSE_ROOT.parent

ASSETS = COURSE_ROOT / "assets"
sys.path.insert(0, str(ASSETS / "helpers"))
import aivision

print("研修フォルダ / course root :", COURSE_ROOT)
print("素材フォルダ / assets      :", ASSETS)

#### カーネルを確認する
このノートブックは **`aivision-anomalib` カーネル** で実行します。
違うカーネルだと `ModuleNotFoundError: No module named 'anomalib'` になります。

その場合は メニューの `Kernel` → `Change Kernel` → `Python 3 (AI Vision Anomalib)` を選び、
**ノートブックを最初から実行し直してください**。

#### Confirm the kernel
Run this notebook with the **`aivision-anomalib`** kernel. With another kernel you get
`ModuleNotFoundError: No module named 'anomalib'`.

Choose `Kernel` -> `Change Kernel` -> `Python 3 (AI Vision Anomalib)` and **re-run from the top**.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import importlib
import sys

print("Python version:", sys.version.split()[0])

REQUIRED = [
    ("anomalib", "anomalib"),
    ("torch", "torch"),
    ("torchvision", "torchvision"),
    ("cv2", "opencv-python-headless"),
    ("lightning", "lightning"),
    ("pandas", "pandas"),
]

missing = []
for import_name, pip_name in REQUIRED:
    try:
        module = importlib.import_module(import_name)
        version = getattr(module, "__version__", "unknown")
        print(f"  [OK]      {pip_name:<20} {version}")
    except ImportError:
        print(f"  [MISSING] {pip_name:<20} -> pip install {pip_name}")
        missing.append(pip_name)

if missing:
    print("\n未導入のパッケージがあります / Some packages are missing:", ", ".join(missing))
    print("docs/troubleshooting/installation.md を確認してください / see that file for help.")
else:
    print("\nこのノートブックに必要なライブラリはすべて揃っています。")
    print("All libraries required by this notebook are available.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
FONT_FAMILY = aivision.configure_japanese_font()
aivision.set_plot_style()

if FONT_FAMILY is None:
    print("日本語フォントが見つかりません。図のタイトルは英字になります。")
    print("No Japanese font found; figure titles fall back to English.")
else:
    print("図の日本語フォント / figure font:", FONT_FAMILY)

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import warnings
warnings.filterwarnings("ignore")     # 依存ライブラリの警告を抑える / silence dependency warnings

import json
from pathlib import Path

import numpy as np
import pandas as pd
import torch
import cv2
import matplotlib.pyplot as plt

import anomalib
from anomalib.data import Folder
from anomalib.models import Patchcore
from anomalib.engine import Engine

print("anomalib   :", anomalib.__version__)
print("torch      :", torch.__version__)
print("CUDA       :", torch.cuda.is_available())
print("device     :", "cuda" if torch.cuda.is_available() else "cpu")

#### 使う API（バージョン固定）
Anomalib は **2.x 系** で API が大きく変わりました。
1.x 系のサンプルコードをコピーすると動きません。

この研修では **Anomalib 2.7.0** で動作確認しています。使うのは 3 つのクラスだけです。

| クラス | 役割 |
|---|---|
| `anomalib.data.Folder` | フォルダ構成からデータを読む |
| `anomalib.models.Patchcore` | PatchCore 本体 |
| `anomalib.engine.Engine` | 学習・評価・推論を実行する |

#### The API used here (pinned version)
Anomalib's API changed substantially in **2.x**; copying 1.x sample code will not work.

This course is verified with **Anomalib 2.7.0**. Only three classes are used:

| Class | Role |
|---|---|
| `anomalib.data.Folder` | reads data from a folder layout |
| `anomalib.models.Patchcore` | the PatchCore model |
| `anomalib.engine.Engine` | runs fit, test and predict |

---

## Part 1　データの構成 / The data layout

#### 正常だけを学習に使う
使うフォルダ構成は次のとおりです。

```text
assets/anomaly_dataset/pvc_sheet/
    train/good/                      <- 学習に使う（正常のみ）
    test/good/                       <- 評価に使う（正常）
    test/black_spot/                 <- 評価に使う（異常）
    test/scratch/
    test/streak/
    test/irregular_mark/
    ground_truth/<種類>/xxx_mask.png <- 正解マスク（評価用）
```

**`train/` には異常画像が 1 枚も入っていません。** これが PatchCore の前提です。

#### Only normal images are used for training
The layout is:

```text
assets/anomaly_dataset/pvc_sheet/
    train/good/                      <- training (normal only)
    test/good/                       <- evaluation (normal)
    test/black_spot/                 <- evaluation (abnormal)
    test/scratch/, test/streak/, test/irregular_mark/
    ground_truth/<type>/xxx_mask.png <- ground-truth masks (evaluation)
```

**`train/` contains no abnormal images at all.** That is the premise of PatchCore.

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
dataset_root = ASSETS / "anomaly_dataset" / "pvc_sheet"
layout_path = dataset_root / "dataset_layout.json"

if not layout_path.is_file():
    raise FileNotFoundError(
        "異常検知データセットがありません。基本環境を有効化した CMD で教材ルートから実行してください / Dataset missing; run in Command Prompt at the course root with the base environment activated:\n"
        "  python assets\\scripts\\prepare_datasets.py --only anomaly\n"
        "/ anomaly dataset missing; run prepare_datasets.py --only anomaly"
    )

layout = json.loads(layout_path.read_text(encoding="utf-8"))
print("dataset root :", dataset_root)
print("counts       :", layout["counts"])
print()
print("train に異常画像が無いことを確認する / confirm train has only normal images:")
print("  train/good/*.png :", len(list((dataset_root / "train" / "good").glob("*.png"))))
print("  train 直下の他のフォルダ / other folders under train:",
      [p.name for p in (dataset_root / "train").iterdir() if p.is_dir() and p.name != "good"])

#### データモジュールを作る
`Folder` に、どのフォルダが何かを教えます。

| 引数 | 意味 |
|---|---|
| `normal_dir` | 学習に使う正常画像のフォルダ |
| `normal_test_dir` | 評価に使う正常画像のフォルダ |
| `abnormal_dir` | 評価に使う異常画像のフォルダ（リスト） |
| `mask_dir` | 正解マスクのフォルダ（リスト） |
| `val_split_mode` / `val_split_ratio` | 評価データを検証と試験に分ける方法と比率 |
| `num_workers` | Windows では 0 が安全 |

`val_split_mode="from_test"` は、**評価用フォルダを「検証用」と「試験用」に分ける」** 指定です。
**半分ずつになるのは `val_split_ratio=0.5` を指定しているから**で、`from_test` だけでは
比率は決まりません（この研修の設定値が 0.5 です）。**しきい値は検証用（この研修では
較正用＝calibration と呼びます）で決め、成績は試験用で測る** ためです。

#### Building the datamodule
`Folder` is told which folder is which.

| Argument | Meaning |
|---|---|
| `normal_dir` | normal images for training |
| `normal_test_dir` | normal images for evaluation |
| `abnormal_dir` | abnormal images for evaluation (list) |
| `mask_dir` | ground-truth mask folders (list) |
| `val_split_mode` / `val_split_ratio` | how the evaluation data is split into val/test, and the ratio |
| `num_workers` | 0 is safe on Windows |

`val_split_mode="from_test"` only says **how** the evaluation folders are split. **The 50/50 split
comes from `val_split_ratio=0.5`**, which this course sets; the mode alone does not fix the ratio.
The purpose is to **choose the threshold on validation (called calibration here) and measure the
score on test**.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
datamodule = Folder(
    name=layout["name"],
    root=str(dataset_root),
    normal_dir=layout["normal_dir"],
    normal_test_dir=layout["normal_test_dir"],
    abnormal_dir=layout["abnormal_dir"],
    mask_dir=layout["mask_dir"],
    train_batch_size=8,
    eval_batch_size=8,
    num_workers=0,                  # Windows のノートブックでは 0 / safe on Windows
    val_split_mode="from_test",     # 評価データを検証と試験に分ける
    val_split_ratio=0.5,
    seed=layout["seed"],
)
datamodule.setup()

print("学習サンプル / train :", len(datamodule.train_data))
print("検証サンプル / val   :", len(datamodule.val_data))
print("試験サンプル / test  :", len(datamodule.test_data))
print()
print("試験データの内訳 / test composition:")
print(datamodule.test_data.samples["label"].value_counts().to_string())

#### 学習データに異常が無いことを確認する
ここは **必ず確認** してください。`train` に異常画像が混ざると、その異常は「正常の見本」として
メモリバンクに入ります。その結果、**同じような見え方の異常のスコアが下がり、見逃しが増えます**。

ただし「混ざったら、その欠陥を二度と検出できない」と決まるわけではありません。コアセットの
選ばれ方、特徴の近傍、位置や周囲の見え方、しきい値によって結果は変わります。
**言えるのは「見逃す危険が増える」ところまで**です。

いちばん確実な直し方は、**混ざった画像を取り除いてメモリバンクを作り直し、手元に残した
較正用・評価用の画像で確かめる** ことです。

#### Verify training data contains no anomalies
**Always check this.** If abnormal images leak into the normal set they enter the memory bank
as normal examples. As a result, **scores for similar-looking anomalies fall and misses increase**.

It is **not** true that one contaminated image makes a defect permanently undetectable: the coreset
selection, the feature neighbourhood, the location and the surrounding context, and the threshold
all change the outcome. What can be said is that **the risk of a miss goes up**.

The reliable fix is to **remove the contaminated images, rebuild the memory bank, and verify the
result on the held-out calibration and evaluation images**.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
train_labels = datamodule.train_data.samples["label"].value_counts().to_dict()
print("train のラベル内訳 / train label counts:", train_labels)
assert len(train_labels) == 1 and list(train_labels)[0] == "normal", (
    "train に正常以外の画像が入っています / training data contains non-normal images"
)
print("OK: 学習データは正常のみです / training data is normal-only")

---

## Part 2　モデルを初期化する / Initialising the model

#### バックボーンの選び方
`Patchcore` の主な引数です。

| 引数 | 意味 | この研修の値 |
|---|---|---|
| `backbone` | 特徴を計算するネットワーク | `resnet18`（CPU 向けに軽い） |
| `layers` | どの層の特徴を使うか | `["layer2", "layer3"]` |
| `pre_trained` | 学習済みの重みを使うか | `True` |
| `coreset_sampling_ratio` | メモリバンクに残す割合 | `0.1`（10 %） |
| `num_neighbors` | 距離計算に使う近傍数 | `9` |

論文の既定は `wide_resnet50_2` ですが、この研修の CPU 環境では計算に時間がかかるため、
既定として `resnet18` を使います。**どちらが良いかはデータと時間で決まります。** 判断するときは、
**同じ較正用（検証）データで指標と所要時間を実測して** 比べてください（演習 2）。
「resnet18 のほうが必ず少し悪い」「wide_resnet50_2 はいつも遅すぎる」と決めつけないでください。

> ImageNet の重みは授業前に `preload_models.py --profile anomalib` で取得します。
> 準備セルの `TIMM_USE_OLD_CACHE=1` が URL キャッシュを優先します。
> キャッシュが無ければこの授業セルはエラーで止まり、自動取得しません。

#### Choosing the backbone
The main `Patchcore` arguments:

| Argument | Meaning | Value here |
|---|---|---|
| `backbone` | network that computes features | `resnet18` (light, CPU friendly) |
| `layers` | which layers to use | `["layer2", "layer3"]` |
| `pre_trained` | use pretrained weights | `True` |
| `coreset_sampling_ratio` | fraction kept in the memory bank | `0.1` (10 %) |
| `num_neighbors` | neighbours used for the distance | `9` |

The paper's default is `wide_resnet50_2`; on this course's CPU-only machine it takes longer to run,
so `resnet18` is the default here. **Which one is better depends on the data and the time budget.**
Compare them by **measuring both the metrics and the runtime on the same calibration (validation)
data** (Exercise 2). Do not assume "resnet18 is always slightly worse" or "wide_resnet50_2 is always
too slow".

Preload ImageNet weights before class with `preload_models.py --profile anomalib`.
Preparation sets `TIMM_USE_OLD_CACHE=1` to prefer the URL cache.
A missing/invalid cache stops this lesson cell explicitly; it does not download automatically.


### コードの読み方 / Reading the code

![](../../assets/svg/teach_patchcore_stages.svg)

図：画像座標と特徴空間を区別します。（計算した合成例・模式図。モデルの実測出力ではありません。）

Figure: Distinguish image coordinates from feature space. (computed synthetic / schematic, not measured model output.)

**気づいてほしいこと**

画像座標と特徴空間を区別します。 入力と変換後の値を指でたどってください。

**What to notice**

Distinguish image coordinates from feature space. Trace the input and transformed values.

**考えてみてください / Try to answer**

正常メモリに異常を入れると、距離による判定はどう変わりそうですか。

What could happen if anomalies enter the normal memory?


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
from timm.models import get_pretrained_cfg
from timm.models._hub import check_cached_file
backbone_cfg = get_pretrained_cfg("resnet18")
if backbone_cfg is None or not backbone_cfg.url or not check_cached_file(backbone_cfg.url):
    raise FileNotFoundError("resnet18 cache missing or invalid; run preload_models.py --profile anomalib before class")
model = Patchcore(
    backbone="resnet18",
    layers=["layer2", "layer3"],
    pre_trained=True,
    coreset_sampling_ratio=0.1,
    num_neighbors=9,
)
print("モデルを初期化しました / model initialised:", type(model).__name__)
print("内部モデル / inner model:", type(model.model).__name__)
print("coreset_sampling_ratio:", model.coreset_sampling_ratio)

# 分割の記録に残すため、実際に使った設定をそのまま保存します。
# Keep the settings we actually used, so the split manifest can record them.
MODEL_SETTINGS = {
    "backbone": "resnet18",
    "layers": ["layer2", "layer3"],
    "pre_trained": True,
    "coreset_sampling_ratio": 0.1,
    "num_neighbors": 9,
}
print("モデル設定 / model settings:", MODEL_SETTINGS)

---

## Part 3　学習（fit）する / Fitting the model

#### PatchCore の「学習」は 1 回で足りる
PatchCore の学習は、**ニューラルネットワークの重みを更新しません**。
やっていることは次の 2 つだけです。

1. 正常画像をバックボーンに通して **特徴を計算する**
2. その特徴を **メモリバンクに保存する**（コアセットで間引く）

**CNN の重みは凍結したまま**で、データで決まるのは **メモリバンク** です。さらに、そのあとの
**較正（しきい値と正規化の統計）** もデータで決まります。保存される状態は、この
「メモリバンク＋較正の統計」です。

同じ画像からは同じ特徴が得られるので、**この合わせ方では 1 回（`max_epochs=1`）で十分**です。
「何度繰り返しても常に無意味」という意味ではありません。**データを変えたら、もう一度合わせ直し、
較正もし直します。** これは第 4 日の YOLO との大きな違いです。

#### One pass is enough for PatchCore's fit
Fitting PatchCore **does not update any neural-network weights**. It only:

1. runs the normal images through the backbone to **compute features**
2. **stores** those features in the memory bank (reduced to a coreset)

**The CNN weights stay frozen.** What the data determine is the **memory bank**, and later the
**calibration** (threshold and normalisation statistics). The stored state is that bank plus those
calibration statistics.

The same images give the same features, so **one pass (`max_epochs=1`) is enough for this fitting
procedure**. That is not a claim that repetition is always meaningless: **when the data change, you
fit again and recalibrate.** This is a key difference from YOLO on Day 4.


### 変わるのはメモリバンクだけ / only the memory bank changes

![](../../assets/diagrams/teach_backbone_vs_memory.png)

図：重みを更新しないバックボーンと、データで決まるメモリバンク（模式図）。

Figure: a frozen backbone and a memory bank determined by the data (schematic).

**気づいてほしいこと**

ResNet-18 の重みは更新されません（凍結）。データで決まるのはメモリバンクで、その後に較正で決まる後処理（正規化の統計としきい値）も保存される状態の一部です。`max_epochs=1` で足りるのは、この手順が 1 周で終わる作りだからです。

**What to notice**

No ResNet-18 weight is updated (they are frozen). The data determines the memory bank, and the post-processing fitted later on calibration (normalisation statistics and the threshold) is also stored state. `max_epochs=1` is enough because this fitting procedure needs one pass.

**考えてみてください / Try to answer**

- 正常画像を増やしたとき、変わるのはバックボーン・メモリバンク・しきい値のどれですか。
- If you add normal images, which changes: the backbone, the memory bank or the threshold?


#### 経過表示を自分で出す理由
Lightning のリッチな経過表示は、Windows の Jupyter で
ごく稀に `RecursionError` を起こすことがあります。この研修では
**表示を切り替えて**、自分で開始・終了のメッセージを出します。
学習そのものの動作は変わりません。

#### Why the progress display is switched off
Lightning's rich progress display can, rarely, raise `RecursionError` inside Jupyter
on Windows. This course **switches it off** and prints its own start/finish messages. The training
itself is unaffected.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import time

results_root = COURSE_ROOT / "training_results" / "day05"
engine = Engine(
    default_root_dir=str(results_root),
    accelerator="cpu",
    devices=1,
    max_epochs=1,
    logger=False,
    enable_progress_bar=False,   # Windows の Jupyter での表示不具合を避ける / avoid a Windows display issue
    enable_model_summary=False,
)

print("学習（fit）を開始します。1〜2 分かかります / fitting now, this takes 1-2 minutes...")
started = time.time()
engine.fit(model=model, datamodule=datamodule)
elapsed = time.time() - started
print()
print(f"学習（fit）が終わりました / fitting finished in {elapsed:.1f} seconds")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# 分割を「あとから証明できる形」で保存する / record the split so it can be proved later
import json

submission_dir = COURSE_ROOT / "submissions" / "day05"
submission_dir.mkdir(parents=True, exist_ok=True)


def relative_images(dataset):
    return sorted(Path(p).resolve().relative_to(dataset_root.resolve()).as_posix()
                  for p in dataset.samples["image_path"])


partitions = {
    "patchcore_train": relative_images(datamodule.train_data),
    "calibration": relative_images(datamodule.val_data),
    "test": relative_images(datamodule.test_data),
}

overlaps = {}
for name_a, files_a in partitions.items():
    for name_b, files_b in partitions.items():
        if name_a >= name_b:
            continue
        shared = sorted(set(files_a) & set(files_b))
        overlaps[f"{name_a} & {name_b}"] = shared
        assert not shared, f"分割が重なっています / partitions overlap: {name_a} & {name_b}: {shared[:3]}"

# partitions の値は「dataset_root からの相対パス」（例 test/black_spot/xxx.png）です。
# パスをそのまま結合してください。パスが違っても同じ絵なら漏れなので、中身のハッシュで比べます。
# The partition values are already relative to dataset_root (e.g. test/black_spot/xxx.png):
# join them directly, and compare content hashes — the same picture under another name is still
# a leak.
import hashlib


def digest(path):
    """ファイルの中身の指紋 / a fingerprint of the file contents."""
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


yolo_hashes = {}
for path in (ASSETS / "yolo_dataset").rglob("*.png"):
    yolo_hashes.setdefault(digest(path), []).append(path.name)

yolo_overlap = sorted(
    name for name in partitions["test"]
    if (dataset_root / name).is_file() and digest(dataset_root / name) in yolo_hashes
)

import anomalib
import torch


manifest = {
    "schema_version": 2,
    "dataset_root": str(dataset_root.resolve()),
    "dataset_name": layout["name"],
    "dataset_seed": layout["seed"],
    "image_size": layout["image_size"],
    "split": {
        "library": f"anomalib=={anomalib.__version__}",
        "val_split_mode": "from_test",
        "val_split_ratio": 0.5,
        "seed": layout["seed"],
        "note": (
            "val_split_mode='from_test' WITH val_split_ratio=0.5 splits the evaluation folders "
            "into validation (used here as the CALIBRATION partition) and test. The ratio, not "
            "the mode alone, sets the sizes: from_test on its own does not imply 50/50. The test "
            "partition is scored once, for the final report."
        ),
    },
    "counts": {name: len(files) for name, files in partitions.items()},
    "partitions": partitions,
    "overlap_check": {name: len(shared) for name, shared in overlaps.items()},
    "yolo_training_overlap": yolo_overlap,
    "model": {
        "class": type(model).__name__,
        **MODEL_SETTINGS,
        "trained_on": "patchcore_train (normal images only)",
        "frozen": (
            "fit does not update the backbone weights; the memory bank and the later calibration "
            "(threshold and normalisation statistics) are what the data determine"
        ),
        "anomalib": anomalib.__version__,
        "torch": torch.__version__,
    },
    "score_convention": (
        "image-level anomaly score on the 0-1 scale produced by anomalib "
        f"{anomalib.__version__}'s PostProcessor with enable_normalization=True. Its statistics "
        "come from the CALIBRATION (validation) partition only: "
        "clip((raw - threshold) / (max - min) + 0.5, 0, 1), so the learned threshold maps to 0.5; "
        "a degenerate range (max == min) falls back to 1 / 0.5 / 0 by comparison with the "
        "threshold, and missing (NaN) statistics leave the raw value unchanged. Image-level and "
        "pixel-level statistics are fitted separately. The raw patch distance is unbounded "
        "(about 9-21 in this run) and is NOT the displayed map maximum. Normalisation statistics "
        "are never fitted on the test partition."
    ),
    "postprocessor": {
        "normalisation_enabled": bool(model.post_processor.enable_normalization),
        "threshold_source": "validation (calibration) partition",
        "statistics": "image-level and pixel-level min/max/threshold are fitted separately",
    },
}

manifest_path = submission_dir / "split_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8")

print("分割 / partitions:", manifest["counts"])
print("重なり / overlaps :", manifest["overlap_check"])
print("YOLO 学習画像との重なり / overlap with YOLO training images:", len(yolo_overlap))
print()
print("保存 / saved:", manifest_path)


### どの状態が変わったか / Which state changed?

| 段階 / Stage | 対象と変化 / Object and change |
|---|---|
| 特徴抽出 / feature extraction | バックボーン重み：固定 / backbone weights: unchanged |
| fit | 正常メモリ：作成 / normal memory: created |
| calibration | 正規化統計・画像しきい値：選択し固定 / normalization statistics and image threshold: selected and frozen |
| inference | 新しい画像：判定だけ、fitに使わない / new image: judge only, never add to fit |

第4日の15 epochの重み最適化とは違います。予想：正常画像を増やすと何を作り直す必要がありますか。
下の保存先でメモリがモデル状態に含まれることを確認し、保存済み数値と今回の実行を区別します。

This is not Day 4's 15 epochs of weight optimization. Predict what must be rebuilt if normal fit
images are added. Check that the checkpoint contains memory; distinguish saved numeric references
from your current run.


#### 保存されたもの
`engine.fit()` は、モデルの状態を `training_results/day05/` 以下に保存します。
**メモリバンクの中身は、この保存ファイルに入っています。**

#### What was saved
`engine.fit()` writes the model state under `training_results/day05/`.
**The memory bank contents are part of that checkpoint.**

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
print("保存先 / output root:", results_root)
for path in sorted(results_root.rglob("*")):
    if path.is_file():
        print(f"  {path.relative_to(results_root)}  ({path.stat().st_size} bytes)")

---

## Part 4　評価する / Evaluating

#### 4 つの指標
Anomalib は 4 つの指標を返します。**この節は較正用（validation）データだけで計算します。**
試験用データの指標は、しきい値を固定したあと Day 5 ノートブック 03 で 1 回だけ出します。

| 指標 | 意味 |
|---|---|
| `image_AUROC` | 画像単位の OK/NG 判別力（1.0 が最高） |
| `image_F1Score` | 画像単位の調和平均（precision と recall のバランス） |
| `pixel_AUROC` | 画素単位で異常領域を当てる力 |
| `pixel_F1Score` | 画素単位の調和平均 |

**AUROC** は「正常と異常を、しきい値を決めずにどれだけ順位付けできたか」を表します。
0.5 がでたらめ、1.0 が完璧な順位付けです。ただし **AUROC が 1.0 でも「どのしきい値でも完璧」**
という意味ではありません。**この較正用データで順位が完璧だった** という意味です。
実際の判定は選んだ 1 つのしきい値で決まるので、**見逃しと誤報の件数は別に数えます**。

較正用データは正常 6 枚・異常 20 枚です。**この枚数で測れる精度には限りがあります。**

#### The four metrics
Anomalib reports four metrics. **This section uses the calibration (validation) partition only.**
The test metrics are produced once, in Day 5 Notebook 03, after the thresholds are frozen:

| Metric | Meaning |
|---|---|
| `image_AUROC` | ability to separate OK/NG per image (1.0 is best) |
| `image_F1Score` | harmonic mean of image-level precision and recall |
| `pixel_AUROC` | ability to localise anomalous pixels |
| `pixel_F1Score` | harmonic mean of pixel-level precision and recall |

**AUROC** describes how well normal and abnormal are **ranked** without fixing a threshold:
0.5 is random, 1.0 is a perfect ranking. AUROC = 1.0 does **not** mean "perfect at every threshold";
it means the ranking was perfect **on this calibration set**. The actual decision uses one chosen
threshold, so **misses and false alarms are counted separately**.

Calibration holds 6 normal and 20 abnormal images. **That sample size limits what you can claim.**

ここで `engine.test()` はライブラリのメソッド名です。渡す `datamodule.val_dataloader()` が評価する画像を決めるため、これは CALIBRATION（較正用）であり、未使用の最終 TEST の評価ではありません。

Here `engine.test()` is the library method name; the supplied `datamodule.val_dataloader()` determines which images are evaluated. This is CALIBRATION, not the held-out TEST evaluation.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# ここで評価するのは **較正用（validation）** だけです。試験用は、しきい値を固定したあと
# Day 5 ノートブック 03 で 1 回だけ評価します。
# Evaluate the CALIBRATION partition only. The test partition is scored once, in
# Day 5 Notebook 03, after the thresholds have been frozen.
calibration_results = engine.test(model=model, dataloaders=datamodule.val_dataloader())

metrics = calibration_results[0] if isinstance(calibration_results, list) else calibration_results
metrics_frame = pd.DataFrame([metrics]).T
metrics_frame.columns = ["calibration_value"]
print(metrics_frame.round(4).to_string())

submission_dir = COURSE_ROOT / "submissions" / "day05"
submission_dir.mkdir(parents=True, exist_ok=True)
metrics_frame.to_csv(submission_dir / "patchcore_calibration_metrics.csv", encoding="utf-8-sig")
print()
print("保存 / saved:", submission_dir / "patchcore_calibration_metrics.csv")
print("試験用の指標は、しきい値を固定したあと Day 5 ノートブック 03 で出します。")
print("Test metrics are produced in Day 5 Notebook 03, after the thresholds are frozen.")

### 実行結果の例 / Expected output

この結果は、この研修で保存した実行の記録です（モデル・データ分割・seed は本文に記載）。
細かな数値は環境で変わりますが、傾向と大小関係を読み取ってください。

```text
                        calibration_value
image_AUROC                        1.0000
image_F1Score                      0.9744
pixel_AUROC                        0.9851
pixel_F1Score                      0.4307

保存 / saved: .../submissions/day05/patchcore_calibration_metrics.csv
試験用の指標は、しきい値を固定したあと Day 5 ノートブック 03 で出します。
```

These numbers come from the run recorded for this course (model, split and seed are stated in the
text). Exact values vary between machines; read the trends and the ordering. These are the
**calibration** (validation) numbers — ResNet-18, `image_size=256`, seed 2026 — and the test numbers
appear only in Notebook 03.

```text
image_AUROC    1.0000
image_F1Score  0.9744
pixel_AUROC    0.9851
pixel_F1Score  0.4307
```


#### 数字が良すぎるときに疑うこと
合成画像は **実際の工場画像より簡単** です。欠陥がはっきりしており、
背景のばらつきも小さいため、`image_AUROC` が 1.0 に近づきます。

**これは「実力」ではありません。** 実際の塩ビでは、色ムラ・照明ムラ・反射などが加わり、
同じ数字は出ません。さらに、較正用は正常 6 枚・異常 20 枚しかないので、
**この枚数で測った数字は、そのまま生産の性能にはなりません**。

この研修の数字を、**製品の検査精度として引用してはいけません。**

#### When the numbers look too good
Synthetic images are **easier than real factory images**: defects are crisp and the
background variation is small, so `image_AUROC` approaches 1.0.

**That is not real-world skill.** Real PVC adds colour mottling, uneven lighting and reflections, and
the numbers will not transfer. Calibration also holds only 6 normal and 20 abnormal images, so a
number measured on that set is **not a production performance figure**.

**Never quote this course's numbers as the inspection accuracy of a product.**


---

## Part 5　異常スコアとヒートマップ / Scores and heatmaps

### 元画像・異常マップ・重ね表示 / original, score map and overlay

![](../../assets/diagrams/teach_score_map_overlay.png)

図：**説明用に作った合成の模式図**（なだらかな山のような形）です。実際の PatchCore の出力ではありません。並べ方の見本として、同じ画像を同じ色スケール（0.0〜0.4）で 3 枚にしています。

Figure: **a synthetic illustration created for teaching** (a smooth hill-like bump); it is not actual PatchCore output. It shows the layout only: the same image three ways on one shared colour scale (0.0-0.4).

**気づいてほしいこと**

比べるときは **表示範囲（vmin と vmax）を固定** します。画像ごとに自動で範囲を変えると、正常品でも最大値が最も明るい色になり、比べられなくなります。このノートブックのヒートマップは、**すべての画像で同じ 0〜1 の範囲** を使い、**カラーバー** を付けて表示します（`MAP_VMIN` と `MAP_VMAX`）。色は **カラーマップと表示範囲** で決まるので、**色の意味はカラーバーで確認**し、明るい部分だけで NG と決めないでください。範囲の外の値は端の色で打ち切られます（**クリッピング**）。

**What to notice**

Fix the **display range (vmin and vmax)** whenever you compare maps. If each image chooses its own
range, even a good part shows its maximum in the brightest colour and the comparison is meaningless.
The heatmaps in this notebook use **one shared 0-1 range** for every image and carry a **colourbar**
(`MAP_VMIN` and `MAP_VMAX`). Colours come from the **colormap and the display range**, so **read the
colourbar** and never decide NG from the brightest region alone. Values outside the range are
clipped to the end colours (**clipping**).

**考えてみてください / Try to answer**

- 表示範囲を画像ごとに変えると、2 枚のマップを比べられなくなるのはなぜですか。
- Why does a per-image display range make two maps impossible to compare?


### 数字を読む順番 / Read the numbers in this order

下の図は計算した合成おもちゃです。較正・試験画像やモデル出力は使いません。
予想：おもちゃの合否を決めるのはどの行ですか。

The panel below is a calculated synthetic toy, using no calibration/test images or model outputs.
Predict: which row determines the toy decision?

| 入力 → 操作 → 値 / Input → operation → value | 用途 / Use |
|---|---|
| 距離の最大6 × 仮の重み0.8 = 4.800 / max distance 6 × chosen weight 0.8 = 4.800 | おもちゃの生の画像スカラー / toy raw image scalar |
| 距離格子 → 3×3平均 → マップ最大2.000 / distance grid → 3×3 mean → map max 2.000 | 場所の手がかり / location clue |
| clip((4.8 − 4) / 8 + 0.5, 0, 1) = 0.600 | 仮の画像しきい値0.50と比較：NG / compare with chosen image threshold 0.50: NG |

5×5の距離格子は全て1、中央は6、上下左右は2です。端を複製して3×3平均を計算します。
重み0.8・中心4・範囲8・しきい値0.50は説明用に選んだ値で、較正で求めた統計ではありません。
これは実際のPatchCore計算の再現ではなく、異なる数の役割の説明です。0.600は確率ではありません。

The 5×5 distance grid contains ones, a centre of 6 and four direct neighbours of 2; use edge padding for the 3×3 mean.
Weight 0.8, centre 4, range 8 and threshold 0.50 are chosen teaching constants, not fitted calibration statistics.
This illustrates distinct numeric roles, not the actual PatchCore algorithm. 0.600 is not a probability.
生の値を0.50と比べてはいけません。表示のvmaxだけを変えると判定は？
Do not compare raw values with 0.50. Does changing only display vmax alter the decision?


### 合成おもちゃの数値マップ / Calculated synthetic toy numeric map

![](../../assets/reference_outputs/teach_numeric_map_toy.png)

図：合成おもちゃの入力・3×3平均マップ・同じ値の重ね表示。`generate_reference_outputs.toy_numeric_map` の固定配列から計算します。実際の較正・TESTデータや推論出力ではありません。

Figure: synthetic toy input, 3×3 mean map and overlay of the same values, calculated from fixed arrays in `generate_reference_outputs.toy_numeric_map`. Not actual calibration/TEST data or inference output.

**気づいてほしいこと**

生の画像スカラー4.800、平滑化マップの最大2.000、変換後の画像スカラー0.600は別物です。同じ尺度の画像しきい値と比べるのは最後の値です。実際の計算は後のコードと較正統計を使います。

**What to notice**

The raw image scalar 4.800, smoothed map maximum 2.000 and transformed image scalar 0.600 differ. Compare the last with an image threshold on the same scale. The later code uses the actual algorithm and fitted calibration statistics.

**考えてみてください / Try to answer**

- マップの最大値と画像スコアが一致しないのはなぜですか。
- Why do the map maximum and the image score differ?


#### 1 枚ずつ結果を見る
`engine.predict()` は、画像ごとの結果を返します。

| 属性 | 内容 |
|---|---|
| `pred_score` | 異常スコア。**後処理が有効で統計が fit 済みのときだけ 0〜1** |
| `pred_label` | しきい値で決めた OK/NG |
| `anomaly_map` | 画素ごとの異常の度合い（後処理で 0〜1 に変換済み） |
| `gt_label` | 正解（異常かどうか） |
| `image_path` | 画像のパス |

`pred_score` が 0〜1 になるのは、**正規化（`enable_normalization=True`）が有効で、
較正用データから最小・最大の統計が入っているとき**です。統計が無い（NaN の）ときは、
Anomalib は **生の距離をそのまま返します**。その値は 0〜1 に収まりません。
「いつも 0〜1 で返ってくる」と決めつけないでください。

#### Look at results image by image
`engine.predict()` returns per-image results:

| Attribute | Content |
|---|---|
| `pred_score` | anomaly score; **0-1 only when postprocessing is enabled and its statistics were fitted** |
| `pred_label` | OK/NG decided by the threshold |
| `anomaly_map` | per-pixel anomaly strength (rescaled to 0-1 by postprocessing) |
| `gt_label` | ground truth (abnormal or not) |
| `image_path` | the image path |

`pred_score` is on 0-1 only when **normalisation (`enable_normalization=True`) is enabled and the
min/max statistics exist from the calibration partition**. When those statistics are missing (NaN),
Anomalib **returns the raw distance unchanged**, which is not bounded to 0-1. Do not assume every
returned value is a 0-1 score.


In [ ]:
# ▸ この値を変更 / Change this value：値を変えて、結果を予想してから実行します。
# ▸ Change this value: edit it below, predict the result, then run.
# 較正用（validation）と試験用（test）の両方のスコアを、ここで 1 回だけ計算して保存します。
# 試験用の値は、しきい値を固定するまで表示も比較もしません（第5日ノートブック03 で使います）。
# Score BOTH the calibration partition and the final test partition once, and store them.
# The test values are neither printed nor compared until the thresholds are frozen
# (Day 5 Notebook 03 uses them).
predictions_by_partition = {
    "calibration": engine.predict(model=model, dataloaders=datamodule.val_dataloader()),
    "test": engine.predict(model=model, dataloaders=datamodule.test_dataloader()),
}
for name, batches in predictions_by_partition.items():
    total = sum(len(batch.pred_score) for batch in batches)
    print(f"{name:<12} batches={len(batches):>2}  images={total}")


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
import json

rows = []
for partition, batches in predictions_by_partition.items():
    for batch in batches:
        scores = batch.pred_score.detach().cpu().numpy()
        labels = batch.pred_label.detach().cpu().numpy().astype(bool)
        gt_labels = batch.gt_label.detach().cpu().numpy().astype(bool)
        for path, score, label, gt in zip(batch.image_path, scores, labels, gt_labels):
            rows.append({
                "partition": partition,
                "image": Path(path).resolve().relative_to(dataset_root.resolve()).as_posix(),
                "score": round(float(score), 4),
                "anomalib_predicted_ng": bool(label),
                "truly_abnormal": bool(gt),
            })

scores_frame = pd.DataFrame(rows)
print(scores_frame.groupby(["partition", "truly_abnormal"]).size().to_string())
print()
print("較正用のスコアは次の節のスライダーで使います。試験用は最後の評価まで表示しません。")
print("Calibration scores feed the slider below. The test scores are not displayed until the end.")

payload = {
    "schema_version": 2,
    "score_convention": (
        "image-level anomaly score on the 0-1 scale produced by anomalib's PostProcessor, "
        "normalised with CALIBRATION (validation) partition statistics "
        "(enable_normalization=True): clip((raw - threshold) / (max - min) + 0.5, 0, 1), so the "
        "learned threshold maps to 0.5. A degenerate range falls back to 1 / 0.5 / 0, and missing "
        "statistics leave the raw value unchanged. Raw patch distances are not bounded to 0-1."
    ),
    "postprocessor_threshold": None,
    "images": rows,
}
try:
    payload["postprocessor_threshold"] = float(model.post_processor.adaptive_threshold)
except Exception:
    payload["postprocessor_threshold"] = None

predictions_path = submission_dir / "patchcore_predictions.json"
predictions_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")
print()
print("保存 / saved:", predictions_path)


#### 数値を見るときの注意
`pred_label` は Anomalib が **較正用データで決めたしきい値** による判定です。
しきい値は `post_processor` に入っており、`model.post_processor` から確認できます。
画像スコア用と画素マップ用のしきい値は **別々** に決まります。

このしきい値は **合成データ用** です。実運用では、**正常品だけを流してスコアの分布を測り、
しきい値を決め直す** 必要があります。ただし、**正常品だけでは「見逃し（再現率）」は測れません**。
正常品で測れるのは **誤報（正常品を NG と言う割合）** までです。見逃しまで測るには、
**異常のラベルが付いた較正用データ** が必要です（第5日ノートブック03 はそうしています）。

> **任意の詳細：スコアの正規化式（anomalib 2.7.0）**
> 画像スコアと画素マップは、別々の統計で `clip((raw - threshold) / (max - min) + 0.5, 0, 1)` に
> 変換されます。`min`・`max`・`threshold` は **較正用（validation）分割** から求めた値です。
> `max - min = 0` のときは、しきい値との比較で `1 / 0.5 / 0` に落とします。統計が無い（NaN の）
> ときは、変換せず生の値を返します。この式のため、**較正で決めたしきい値は 0〜1 の目盛りでは
> 0.5** になります。

#### Caution when reading the numbers
`pred_label` uses **the threshold Anomalib chose on the calibration data**, stored in the
`post_processor` and readable via `model.post_processor`. The image-level and pixel-level thresholds
are chosen **separately**.

That threshold is **for this synthetic data only**. In production you would **run known-good
products, measure the score distribution and set the threshold again**. Normal images alone,
however, can only tell you about **false alarms**, not about **misses (recall)**; measuring recall
needs **labelled abnormal calibration images**, which is what Day 5 Notebook 03 does.

> **Optional detail: the normalisation formula (anomalib 2.7.0)**
> Image scores and pixel maps are rescaled separately with
> `clip((raw - threshold) / (max - min) + 0.5, 0, 1)`, where the `min`, `max` and `threshold` come
> from the **calibration (validation)** partition. When `max - min = 0`, the value falls back to
> `1 / 0.5 / 0` by comparison with the threshold; when the statistics are missing (NaN), the raw
> value is returned unchanged. Because of this formula, **the threshold chosen on calibration sits
> at 0.5 on the 0-1 scale**.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# 見るのは **較正用データ** の分布だけです。試験用データの分布は最後まで見ません。
# Look only at the CALIBRATION distribution. The test distribution stays unseen.
calibration = scores_frame[scores_frame["partition"] == "calibration"]
normal_scores = calibration.loc[~calibration["truly_abnormal"], "score"]
abnormal_scores = calibration.loc[calibration["truly_abnormal"], "score"]

print("較正用・正常品のスコア / calibration normals :",
      f"n={len(normal_scores)}  min={normal_scores.min():.4f}  max={normal_scores.max():.4f}")
print("較正用・異常品のスコア / calibration abnormals:",
      f"n={len(abnormal_scores)}  min={abnormal_scores.min():.4f}  max={abnormal_scores.max():.4f}")
print()
print("この 2 つの間にしきい値を置けるかどうかが、この手法の実力です。")
print("Whether a gap exists between them is what this method can actually do.")
print()
print("注意 / caution: 試験用データのスコアは、まだ見ないでください。")
print("Do not look at the test scores yet.")

In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# 較正用（validation）のバッチだけを表示します。試験用はしきい値を固定するまで見ません。
# 後処理済みマップは 0〜1 なので、表示範囲も全画像で 0〜1 に固定します。
# Show CALIBRATION batches only; the test partition stays unseen until the thresholds are frozen.
# The postprocessed maps are on 0-1, so one display range (0-1) is used for every image.
MAP_VMIN, MAP_VMAX = 0.0, 1.0


def show_heatmaps(batch, count=4):
    images = batch.image.detach().cpu().numpy()          # (N, 3, H, W), 0-1
    maps = batch.anomaly_map.detach().cpu().numpy()      # (N, H, W), postprocessed 0-1
    scores = batch.pred_score.detach().cpu().numpy()
    truths = batch.gt_label.detach().cpu().numpy()

    count = min(count, len(images))
    fig, axes = plt.subplots(count, 3, figsize=(11, 3.1 * count))
    if count == 1:
        axes = axes.reshape(1, -1)

    plotted = None
    for index in range(count):
        rgb = np.transpose(images[index], (1, 2, 0))
        heat = maps[index]
        axes[index, 0].imshow(rgb); axes[index, 0].set_title("image", fontsize=8)
        plotted = axes[index, 1].imshow(heat, cmap="inferno", vmin=MAP_VMIN, vmax=MAP_VMAX)
        axes[index, 1].set_title(
            f"anomaly map (fixed {MAP_VMIN:.0f}-{MAP_VMAX:.0f})", fontsize=8)
        axes[index, 2].imshow(rgb)
        axes[index, 2].imshow(heat, cmap="inferno", vmin=MAP_VMIN, vmax=MAP_VMAX, alpha=0.5)
        axes[index, 2].set_title(
            f"overlay  score={scores[index]:.3f}  truth={'NG' if truths[index] else 'OK'}", fontsize=8)
        for ax in axes[index]:
            ax.set_xticks([]); ax.set_yticks([])
    plt.tight_layout()
    colourbar = fig.colorbar(plotted, ax=axes.ravel().tolist(), fraction=0.02, pad=0.01)
    colourbar.set_label("postprocessed anomaly value (0.5 = calibration threshold)", fontsize=7)
    plt.show()


for batch in predictions_by_partition["calibration"][:1]:
    show_heatmaps(batch, count=4)

print("表示範囲は 0〜1 で固定です。範囲外の値は端の色で打ち切られます（クリッピング）。")
print("The display range is fixed at 0-1; values outside it are clipped to the end colours.")
print("ここに表示するのは較正用の画像です。試験用はしきい値を固定するまで表示しません。")
print("These are calibration images; the test partition is not shown before the freeze.")


#### ヒートマップと数値マップを保存する
第 5 日ノートブック 03 で、YOLO の結果と並べて表示するために、
ヒートマップを画像ファイルとして保存します。

- **重ね表示の画像（`heatmaps/`）** は **較正用データ** から作ります。しきい値を固定する前に
  目で見てよいのは較正用だけだからです。
- **数値マップ（`maps/*.npz`）** は、あとから教材図を作り直せるように、評価用画像の数値も
  そのまま保存します。**保存すること** と **見て判断すること** は別です。この節では中身を
  表示しません。

#### Save the heatmaps as images
Save the heatmaps as image files so Day 5 Notebook 03 can show them next to the YOLO
results.

- The **overlay pictures (`heatmaps/`)** come from **calibration images**: only the calibration
  partition may be inspected before the thresholds are frozen.
- The **numeric maps (`maps/*.npz`)** are written as numbers for the evaluation images too, so the
  teaching figures can be rebuilt later. **Saving** them is not the same as **looking at** them;
  nothing in this section displays their contents.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
heatmap_dir = submission_dir / "heatmaps"
heatmap_dir.mkdir(parents=True, exist_ok=True)
# 数値マップは別フォルダに置きます（描画済み画像と混ぜない）
# keep the numeric maps apart from pictures
map_dir = submission_dir / "maps"
map_dir.mkdir(parents=True, exist_ok=True)
# 表示範囲は前のセルと同じ 0〜1 に固定します / same fixed display range as the cell above
MAP_VMIN, MAP_VMAX = 0.0, 1.0

def save_overlay(batch, index, out_path):
    """1 枚分のヒートマップを重ねた画像を保存する / save one overlay image."""
    image = np.transpose(batch.image[index].detach().cpu().numpy(), (1, 2, 0))
    heat = batch.anomaly_map[index].detach().cpu().numpy()
    figure = plt.figure(figsize=(4, 4))
    plt.imshow(image)
    plt.imshow(heat, cmap="inferno", vmin=MAP_VMIN, vmax=MAP_VMAX, alpha=0.5)
    plt.axis("off")
    plt.tight_layout()
    figure.savefig(out_path, dpi=110, bbox_inches="tight")
    plt.close(figure)


def save_map(batch, index, out_path):
    """生のパッチ距離マップを数値（npz）で保存する / save the raw patch-distance map as numbers.

    描画した画像ではなく数値を残すと、あとから教材図を数値で作り直せます。
    Keeping numbers (not a rendered picture) lets the teaching figures be rebuilt from data.
    """
    with torch.no_grad():
        raw = model.model(batch["image"])
    maps = raw.anomaly_map.detach().cpu().numpy()
    if maps.ndim == 4:
        maps = maps.squeeze(1)
    scores = raw.pred_score.detach().cpu().numpy().reshape(-1)
    np.savez_compressed(
        out_path,
        anomaly_map=maps[index].astype("float32"),
        raw_score=np.float32(scores[index]),
        truly_abnormal=np.bool_(bool(batch.gt_label[index])),
        image=Path(batch.image_path[index]).resolve().relative_to(dataset_root.resolve()).as_posix(),
    )


# 数値マップは評価用の全枚分を保存します（描画画像より小さく、あとから図を作り直せる）。
# Save a numeric map for every evaluation image: it is small and lets the figures be rebuilt later.
# 保存は「見る」ことと同じではありません。この節では中身を表示しません。
# Saving is not looking: nothing here displays the contents.
for batch in predictions_by_partition["test"]:
    for index, path in enumerate(batch.image_path):
        name = Path(path).stem
        save_map(batch, index, map_dir / f"{name}_map.npz")

# 人が見る重ね表示は較正用データだけから作ります（まだしきい値を固定していないため）。
# The overlays people look at come from CALIBRATION images only: the thresholds are not frozen yet.
saved_heatmaps = []
for batch in predictions_by_partition["calibration"]:
    for index in range(len(batch.pred_score)):
        truth = bool(batch.gt_label[index])
        score = float(batch.pred_score[index])
        # 重ね表示は異常 2 枚・正常 1 枚までにする / keep only 2 abnormal and 1 normal overlay
        if truth and sum(1 for h in saved_heatmaps if h["truth"]) >= 2:
            continue
        if (not truth) and sum(1 for h in saved_heatmaps if not h["truth"]) >= 1:
            continue
        name = Path(batch.image_path[index]).stem
        out_path = heatmap_dir / f"{name}_heatmap.png"
        save_overlay(batch, index, out_path)
        saved_heatmaps.append({"file": out_path.name, "partition": "calibration",
                               "score": round(score, 4), "truth": truth})

print(pd.DataFrame(saved_heatmaps).to_string(index=False))
print()
print("数値マップ / numeric maps:", len(list(map_dir.glob("*_map.npz"))), "files")
print("重ね表示は較正用データだけから作っています（試験用はしきい値固定後に第5日ノートブック03 で見ます）。")
print("The overlays come from calibration images only; the test partition appears in "
      "Day 5 Notebook 03 after the freeze.")
print("この数値マップは assets/scripts/generate_reference_outputs.py が教材図に使います。")
print("These numeric maps are what generate_reference_outputs.py draws the teaching figure from.")


#### ヒートマップの読み方
色は **表示範囲とカラーマップ** で決まります。この節の図は後処理済みの値なので、
**0〜1 の固定した範囲** で表示し、0.5 が較正で決めた画素しきい値です。
範囲を超えた値は端の色で打ち切られます（クリッピング）。**カラーバーで数値を確認**してください。

- **NG の画像** でも、欠陥が明るい点として出るとは限りません。広く薄く出ることも、
  暗いままのこともあります。
- **OK の画像** でも、ムラの部分がやや明るく出ることがあります。

だから、**明るい点があるかどうかで判定しない**でください。判定は画像スコアとしきい値で行い、
ヒートマップは「どこを見ているか」を確かめるために使います。もし OK の画像で明るい場所が出たら、
それは **「正常のばらつきを異常と判定している」** 可能性があります。学習データに、そのばらつきを
追加する必要があります。

#### How to read a heatmap
The colours come from the **display range and the colormap**. The maps here are postprocessed, so
they are shown on a **fixed 0-1 range**, where 0.5 is the pixel threshold chosen during calibration.
Values outside the range are clipped to the end colours. **Read the colourbar**, not the impression.

- **abnormal images** do not always show a bright spot: a defect can appear as a broad faint patch,
  or the map can stay dark.
- **normal images** can show slightly bright mottled areas.

So **do not decide from the presence of a bright spot**. The decision uses the image score and the
threshold; the heatmap tells you where the model is looking. If a normal image shows a bright
region, the model is probably **calling acceptable variation an anomaly**; add that variation to the
training data.


In [ ]:
# ▸ 実行のみ / Run only：このまま実行します。書き換えません。
# ▸ Run only: press Shift + Enter and leave the code as it is.
# 提出用にスコア一覧を保存する / save the score table for submission
scores_path = submission_dir / "patchcore_scores.csv"
scores_frame.to_csv(scores_path, index=False, encoding="utf-8-sig")

# すでに保存した v2 の予測ファイルに、評価指標を追記します（作り直しません）。
# Add the metrics to the v2 payload that was already written; do not rewrite the schema here.
payload = json.loads(predictions_path.read_text(encoding="utf-8"))
payload["metrics"] = {k: float(v) for k, v in metrics.items()}
payload["model"] = manifest["model"]
predictions_path.write_text(json.dumps(payload, indent=2, ensure_ascii=False), encoding="utf-8")

print("保存 / saved:", scores_path)
print("保存 / saved:", predictions_path)
print("schema_version:", payload["schema_version"])

---

## 演習 / Exercises

#### 演習 1（基本）：指標を記録する
Part 4 の 4 つの指標を表に記入してください。
`image_AUROC` と `pixel_AUROC` の違いも、1 行で説明してください。

#### Exercise 1 (basic): record the metrics
Record the four metrics from Part 4 and explain in one line how `image_AUROC` differs
from `pixel_AUROC`.
Done when all four values are recorded and the difference is explained.

In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 1 / Exercise 1
record = {
    "image_AUROC": None,
    "image_F1Score": None,
    "pixel_AUROC": None,
    "pixel_F1Score": None,
    "difference": "",
}
for key, value in record.items():
    print(f"{key:<16} {value if value not in (None, '') else '(未記入 / not written)'}")

#### 演習 2（中級）：設定を変えて較正用データで比べる
`backbone="wide_resnet50_2"` に変えて、**較正用（検証）データで** 指標と所要時間を測り、
`resnet18` と比べてください。CPU しか使えない環境で `wide_resnet50_2` が遅すぎるときは、
**別の設定**（例：`layers=["layer3"]`、`coreset_sampling_ratio` の変更）で同じ比較をして
かまいません。**重いバックボーンを使うかどうかは任意** です。この演習の目的は
「同じ較正用データで、指標と所要時間を実測して比べ、記録すること」です。

- 比べるのは **同じ較正用データ** での `image_AUROC` などの指標と **実測の所要時間** です。
- 試験用データは、最後に選んだ 1 つの設定を評価するために残しておきます。
- 記録には **分割（calibration）・モデル（backbone）・実行の識別子** を必ず書きます。

#### Exercise 2 (intermediate): change a setting and compare on calibration
Switch to `backbone="wide_resnet50_2"`, measure the metrics **on the calibration (validation)
loader** and compare them with `resnet18`, including the measured runtime. When `wide_resnet50_2` is
too slow on a CPU-only machine, compare **another setting** instead (for example
`layers=["layer3"]` or a different `coreset_sampling_ratio`). **Using the heavier backbone is
optional**; the point of the exercise is to **measure and record the metrics and the runtime of two
configurations on the same calibration data**.

- Compare on **the same calibration data**, using metrics such as `image_AUROC` plus the
  **measured runtime**.
- Keep the test partition for evaluating the single configuration you finally choose.
- Label every recorded number with the **split (calibration), the model (backbone) and the run
  identity**.
Done when both configurations are recorded on calibration and you state which one you would choose.


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 2 / Exercise 2
import time

comparison = []
# 例 / example（較正用データで比べます / compare on the calibration loader）:
# started = time.time()
# model_wide = Patchcore(backbone="wide_resnet50_2", layers=["layer2","layer3"], pre_trained=True)
# engine_wide = Engine(default_root_dir=str(results_root / "wide"), accelerator="cpu", devices=1,
#                      max_epochs=1, logger=False)
# engine_wide.fit(model=model_wide, datamodule=datamodule)
# wide_metrics = engine_wide.test(model=model_wide, dataloaders=datamodule.val_dataloader())
# comparison.append({"backbone": "wide_resnet50_2", "split": "calibration",
#                    "image_AUROC": float(wide_metrics[0]["image_AUROC"]),
#                    "seconds": round(time.time() - started, 1)})
# comparison.append({"backbone": "resnet18", "split": "calibration",
#                    "image_AUROC": float(metrics["image_AUROC"]), "seconds": None})

print(pd.DataFrame(comparison or [{"backbone": "(未実施 / not run)", "split": "-",
                                   "image_AUROC": None, "seconds": 0.0}]).to_string(index=False))


#### 演習 3（発展・任意）：正常品のばらつきを増やす
`assets/scripts/pvc_synth.py` を使って、**明るさのムラを大きくした正常画像** を
10 枚追加で生成し、`train/good/` に置いてください。そのうえで再学習し、
**較正用データ（同じ固定したセット）での正常スコアの最大値** が下がったかを確認してください。

#### Exercise 3 (optional challenge): add normal variation
Using `assets/scripts/pvc_synth.py`, generate ten extra normal images with **stronger
brightness variation**, put them in `train/good/`, fit again, and check whether the **maximum normal
score on the same fixed calibration set** falls.
Done when ten extra images exist, fitting has been re-run, and the normal-score maximum is compared before and after on the same calibration set.


In [ ]:
# ▸ 自分でコードを書く / Write your code：空欄を自分で埋めてから実行します。
# ▸ Write your code: fill in the blanks, then run.
# 演習 3 / Exercise 3
import sys
sys.path.insert(0, str(ASSETS / "scripts"))
import pvc_synth as ps

extra_dir = dataset_root / "train" / "good"
before_max = float(normal_scores.max())
print("学習前の正常スコア最大 / before:", round(before_max, 4))

# 例 / example:
# for i in range(10):
#     image = ps.render_normal(256, 256, seed=90_000 + i, shading_strength=18.0, grain_strength=8.0)
#     ps.save_rgb(image, extra_dir / f"extra_variation_{i:03d}.png")
# print("追加しました / added:", len(list(extra_dir.glob('extra_variation_*.png'))))
print("追加したら、Part 3 の fit から再実行してください / then re-run from the Part 3 fit cell")

---

#### 理解度チェック / Understanding questions

1. なぜ `train` に異常画像を入れてはいけないのですか。
2. PatchCore の「学習」は、何をしていますか。重みは更新されますか。
3. `image_AUROC` と `pixel_AUROC` は、何を測っていますか。
4. `val_split_mode="from_test"` にした理由は何ですか。
5. `image_AUROC` が 1.0 でも、製品の検査精度として引用してはいけないのはなぜですか。

1. Why must abnormal images never appear in `train`?
2. What does PatchCore's "training" actually do? Are weights updated?
3. What do `image_AUROC` and `pixel_AUROC` measure?
4. Why was `val_split_mode="from_test"` used?
5. Why must an `image_AUROC` of 1.0 not be quoted as the inspection accuracy of a product?
Model answers: `docs/instructor/day05_instructor_guide.md`.

---

## 困ったときは / Troubleshooting

エラーが出たら、まず「どのセルで」「どの行で」「どんなメッセージが出たか」を 書き留めてください。提出物にも記録します。

When an error appears, first write down *which cell*, *which line*, and *what the message says*. You will also record it in your submission.

| 症状（日本語） | Symptom (English) | 対処（日本語） | What to do (English) |
| --- | --- | --- | --- |
| `ModuleNotFoundError: No module named 'anomalib'` | `ModuleNotFoundError: No module named 'anomalib'` | カーネルが違います。`Kernel` → `Change Kernel` → `Python 3 (AI Vision Anomalib)`。 | Wrong kernel. Choose `Python 3 (AI Vision Anomalib)`. |
| `AttributeError: 'Folder' object has no attribute 'val_data'` | `AttributeError: 'Folder' object has no attribute 'val_data'` | `val_split_mode="none"` にすると発生します。`from_test` か `from_train` を指定してください。 | Happens with `val_split_mode="none"`. Use `from_test` or `from_train`. |
| `MisMatchError` が出る | `MisMatchError` is raised | 異常画像とマスクの対応がずれています。`ground_truth/<種類>/` のファイル名を確認してください。 | Abnormal images and masks do not pair up. Check the file names under `ground_truth/<type>/`. |
| バックボーンのダウンロードに失敗する | The backbone download fails | 授業前に `preload_models.py --profile anomalib` で取得・検査します。不足は講師に相談してください。 | Run `preload_models.py --profile anomalib` before class; report missing cache to the instructor. |
| 学習が異常に遅い | Training is very slow | `wide_resnet50_2` は CPU では非常に遅いです。`resnet18` を使ってください。 | `wide_resnet50_2` is very slow on CPU; use `resnet18`. |
| メモリ不足になる | Out of memory | `train_batch_size` と `eval_batch_size` を 4 に下げてください。 | Lower `train_batch_size` and `eval_batch_size` to 4. |

## 完了チェックリスト / Completion checklist

**必須の項目だけ** を並べています。**演習 3（発展）は任意** で、未実施でも完了です。時間が余ったときだけ取り組んでください。

Only the **required** items are listed. **Exercise 3 (challenge) is optional**; leaving it undone still counts as complete. Do it only if you have spare time.

すべてにチェックが付いたら、このノートブックは完了です。

This notebook is complete when every box is checked.

- [ ] `aivision-anomalib` カーネルで実行した
      - [ ] Ran with the `aivision-anomalib` kernel
- [ ] `train` が正常のみであることを確認した
      - [ ] Confirmed `train` holds normal images only
- [ ] PatchCore の fit を実行した
      - [ ] Ran the PatchCore fit
- [ ] 4 つの指標を記録した
      - [ ] Recorded the four metrics
- [ ] ヒートマップを確認した
      - [ ] Inspected the heatmaps
- [ ] スコア一覧と予測 JSON を保存した
      - [ ] Saved the score table and prediction JSON
- [ ] 演習 1〜2（必須）を実施した
      - [ ] Completed exercises 1-2 (required)
- [ ] 理解度チェックの 5 問に答えた
      - [ ] Answered the 5 questions
- [ ] ノートブックを保存した
      - [ ] Saved the notebook

## さらに学ぶ / Further learning（任意 / optional）

### 参考リンク / Reference links

- [Anomalib ドキュメント](https://anomalib.readthedocs.io/)
- [Anomalib: Folder データモジュール](https://anomalib.readthedocs.io/en/latest/markdown/guides/reference/data/image/folder.html)
- [PatchCore 論文](https://arxiv.org/abs/2106.08265)
- [torchvision のモデル一覧](https://pytorch.org/vision/stable/models.html)

### 追加チャレンジ / Extra challenge（必須ではありません / not mandatory）

`coreset_sampling_ratio` を 0.01 と 0.5 で試し、メモリ使用量と指標の変化を比べてください。

Try `coreset_sampling_ratio` at 0.01 and 0.5 and compare memory use and metrics.